# Défi 2 — Survie à 2 mois avec une forêt aléatoire

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s2-defi.ipynb)

Patients de réanimation de l'étude **SUPPORT2** : prédire s'ils sont
vivants (`alive`) ou décédés (`dead`) **2 mois** après l'admission, à partir
de 28 variables mesurées à l'admission. Le score est l'**exactitude** ;
prédire `alive` pour tout le monde donne déjà environ 0,59.

Méthode de la séance 2, dans le bon ordre :

1. séparer apprentissage et **validation** ;
2. écrêter les valeurs aberrantes (bornes IQR de l'apprentissage) ;
3. préparer : imputation (+ indicateurs de valeur manquante), one-hot ;
4. entraîner une **forêt aléatoire**, puis régler quelques
   hyperparamètres par **validation croisée** ;
5. mesurer **une fois** sur la validation, puis entraîner sur tout
   l'apprentissage et écrire `submission.csv`.

Soumission : [ml-arena.com](https://ml-arena.com/viewchallenge/200), page du défi 2.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

URL = 'https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/support2/'
X = pd.read_csv(URL + 'X_train.csv', index_col='patient_id')
y = pd.read_csv(URL + 'y_train_survie.csv', index_col='patient_id')['outcome']
X_test = pd.read_csv(URL + 'X_test.csv', index_col='patient_id')
assert (X.index == y.index).all()

colonnes_num = X.select_dtypes(include='number').columns.tolist()
colonnes_cat = X.select_dtypes(exclude='number').columns.tolist()
print(f"Apprentissage : {X.shape}, test : {X_test.shape}")
print(f"{len(colonnes_num)} variables numériques, catégorielles : {colonnes_cat}")
print(y.value_counts(normalize=True).round(3))

Les valeurs manquantes sont nombreuses : jusqu'à 61 % pour `adlp`
(autonomie déclarée par le patient).

In [ ]:
X.isna().mean().sort_values(ascending=False).head(10).round(2)

## 1. Séparer apprentissage et validation

On met de côté 20 % des patients pour la validation, **avant** toute
préparation. `stratify=y` garde la même proportion de décès des deux
côtés.

In [ ]:
X_app, X_val, y_app, y_val = train_test_split(X, y, test_size=0.2,
                                              random_state=42, stratify=y)
print(X_app.shape, X_val.shape)

## 2. Écrêter les valeurs aberrantes

Complétez `bornes_iqr` : elle renvoie, pour chaque colonne numérique, les
bornes $Q_1 - 1.5\,\mathrm{IQR}$ et $Q_3 + 1.5\,\mathrm{IQR}$ calculées
sur le tableau reçu. `ecreter` applique ces bornes à n'importe quel tableau
(`clip(..., axis=1)` aligne les bornes sur les colonnes). Les bornes se
calculent sur **l'apprentissage seul**.

In [ ]:
def bornes_iqr(tableau):
    # TODO : q1, q3 = quantiles 0.25 et 0.75 de tableau[colonnes_num]
    q1 = ...
    q3 = ...
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr


def ecreter(tableau, bas, haut):
    tableau = tableau.copy()
    tableau[colonnes_num] = tableau[colonnes_num].clip(bas, haut, axis=1)
    return tableau


bas, haut = bornes_iqr(X_app)
X_app_e, X_val_e = ecreter(X_app, bas, haut), ecreter(X_val, bas, haut)

In [ ]:
assert (X_app_e['crea'].max() <= haut['crea']) and X_app_e['crea'].isna().sum() == X_app['crea'].isna().sum()
print(f"crea : max {X_app['crea'].max():.1f} → {X_app_e['crea'].max():.1f} mg/dL")
print("✅ Écrêtage correct")

## 3. La préparation : imputation et one-hot

Un `ColumnTransformer` applique une préparation par groupe de colonnes :

- numériques : `SimpleImputer(strategy='median', add_indicator=True)` ;
  l'indicateur ajoute une colonne 0/1 par variable incomplète, car
  l'absence d'une valeur est une information ;
- catégorielles : `SimpleImputer(strategy='most_frequent')` puis
  `OneHotEncoder(handle_unknown='ignore')`.

Il n'est pas encore ajusté : il le sera dans le pipeline, sur
l'apprentissage seul.

In [ ]:
# TODO : complétez les deux préparations
preparation = ColumnTransformer([
    ('num', ..., colonnes_num),
    ('cat', make_pipeline(..., ...), colonnes_cat),
])

## 4. Une forêt aléatoire

Le `Pipeline` enchaîne la préparation et la forêt : `fit` ajuste les deux
sur l'apprentissage, `predict` applique la même préparation à de nouveaux
patients. Entraînez-le sur `X_app_e` et mesurez l'exactitude sur `X_val_e`.

In [ ]:
foret = Pipeline([
    ('prep', preparation),
    ('foret', RandomForestClassifier(n_estimators=200, random_state=42)),
])
# TODO : entraînez foret, puis calculez son exactitude de validation
...

Deux points de comparaison : toujours prédire `alive`, et le modèle du
défi 1 (régression logistique standardisée, ici avec la même préparation).

In [ ]:
print(f"Toujours 'alive' : {(y_val == 'alive').mean():.3f}")
logistique = Pipeline([('prep', preparation), ('echelle', StandardScaler()),
                       ('logistique', LogisticRegression(max_iter=2000))])
logistique.fit(X_app_e, y_app)
print(f"Régression logistique : {logistique.score(X_val_e, y_val):.3f}")

## 5. Régler quelques hyperparamètres

Une petite grille, évaluée en validation croisée à 5 plis **sur
l'apprentissage seul**. Dans un pipeline, un hyperparamètre se nomme
`<étape>__<paramètre>` : `foret__max_depth`. 3 × 2 × 2 = 12 combinaisons,
60 entraînements (environ une minute sur Colab).

In [ ]:
grille = {
    'foret__max_depth': [None, 8, 16],
    'foret__min_samples_leaf': [1, 5],
    'foret__max_features': ['sqrt', 0.3],
}
# TODO : GridSearchCV(foret, grille, cv=5, scoring='accuracy', n_jobs=-1), puis fit
recherche = ...

Mesure **unique** sur la validation, mise de côté depuis le début :
c'est l'estimation honnête du score à attendre sur ML-Arena.

In [ ]:
# TODO : exactitude de recherche.best_estimator_ sur X_val_e
...

## 6. Le modèle final et la soumission

Les hyperparamètres sont choisis : on réentraîne le pipeline sur **tous**
les patients d'apprentissage (bornes IQR recalculées sur `X`), on prédit
les 1541 patients du test, et on écrit `submission.csv` avec les colonnes
`patient_id,outcome`.

In [ ]:
bas, haut = bornes_iqr(X)
modele_final = recherche.best_estimator_
# TODO : entraînez modele_final sur ecreter(X, bas, haut) et y,
# puis prédisez ecreter(X_test, bas, haut)
predictions = ...

soumission = pd.DataFrame({'patient_id': X_test.index, 'outcome': predictions})
soumission.to_csv('submission.csv', index=False)

In [ ]:
verif = pd.read_csv('submission.csv')
assert list(verif.columns) == ['patient_id', 'outcome'], verif.columns
assert len(verif) == 1541 and set(verif['patient_id']) == set(X_test.index)
assert set(verif['outcome']) <= {'alive', 'dead'}
print(verif['outcome'].value_counts())
print("✅ submission.csv est prêt")

Téléchargez le fichier (sur Colab, la cellule suivante l'enregistre dans
vos Téléchargements), puis déposez-le sur la page du défi :
[https://ml-arena.com/viewchallenge/200](https://ml-arena.com/viewchallenge/200).

In [ ]:
try:
    from google.colab import files
except ImportError:
    print("Hors de Colab : submission.csv est dans le dossier du notebook.")
else:
    files.download('submission.csv')